# 03 Applicant lists: snapshot and profile (T2.5a)

**Task:** T2.5a. Profiles the public applicant lists downloaded by
`harborfund.ingest.applicants` (MARAD PIDP, DOT TIGER–BUILD, DOT INFRA) and
`harborfund.ingest.openfema` (OpenFEMA HMA Subapplications v2, which holds BRIC and PDM).

**Reads:** `data/raw/applicants/<source>/<date>/`, `data/raw/openfema/<date>/`,
`data/manifest.csv`, `data/reference/coastal_counties.csv`,
`data/reference/program_inventory.csv`, and T1.1's national 97.047 USAspending pull
(`data/raw/usaspending/<date>/aln_97.047/`) for the award-ID cross-check.

**Writes:** nothing except through `harborfund.ingest.openfema.process`, which builds
`data/processed/bric_pacific_subapplicants.csv` for T1.3 (not committed).

Outputs are kept: they are part of the deliverable (the profile and the checks).

In [1]:
import json

import pandas as pd

from harborfund import config
from harborfund.ingest import applicants, openfema

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
PACIFIC = ["CA", "OR", "WA"]

## 1. Downloads logged in the manifest

T2.5a rows only. `changed_since_t04` compares with the checksums T0.4 recorded on 2026-11-17.

In [2]:
man = pd.read_csv(config.MANIFEST_PATH, dtype=str)
notes = man["notes"].map(json.loads)
t25 = man[notes.map(lambda n: n.get("task") == "T2.5a")].copy()
n = t25["notes"].map(json.loads)
t25["file"] = n.map(lambda d: d.get("file", "").rsplit("/", 1)[-1])
t25["bytes"] = n.map(lambda d: d.get("bytes"))
t25["rows"] = n.map(lambda d: d.get("rows"))
t25["pages"] = n.map(lambda d: d.get("pages"))
t25["changed_since_t04"] = n.map(lambda d: d.get("changed_since_t04"))
t25["page_last_updated"] = n.map(lambda d: d.get("page_last_updated"))
t25["sha256_12"] = t25["sha256"].str[:12]
t25[["source_id", "file", "retrieved_utc", "bytes", "rows", "pages", "sha256_12", "changed_since_t04", "page_last_updated"]]

,source_id,file,retrieved_utc,bytes,rows,pages,sha256_12,changed_since_t04,page_last_updated
17,openfema,HmaSubapplications_v2.csv,2026-11-18T16:18:56Z,28470485,63640.0,None,3a2a89be7b4b,None,None
18,applicants_marad_pidp,FY19-26 PIDP Eligible Applicants.xlsx,2026-11-18T17:42:21Z,75694,1040.0,None,6d910c1c8a12,False,None
19,applicants_dot_tiger_build,TIGER 2009 - BUILD 2025 Eligible Applications.xlsx,2026-11-18T17:42:27Z,1275598,14621.0,None,70ec6e0efd09,False,None
20,applicants_dot_infra,FY 2022 USDOT Discretionary Grant Applicant List.xlsx,2026-11-18T17:42:31Z,210074,3032.0,None,4783fb2c73ca,None,None
21,applicants_dot_infra,INFRA FY22 Applications.xlsx,2026-11-18T17:42:58Z,42154,329.0,None,c51d5da7a58d,None,None
22,applicants_dot_infra,INFRA FY23-24 Applications.xlsx,2026-11-18T17:43:02Z,40824,191.0,None,5945c75f0216,None,None
23,applicants_dot_infra,INFRA Applications 25-26.xlsx,2026-11-18T17:43:05Z,38314,174.0,None,16259a34a5b9,None,None
24,applicants_dot_infra,INFRA FY26 Applications (1).xlsx,2026-11-18T17:43:09Z,29691,88.0,None,f3ed2b5db66e,None,None
25,applicants_dot_infra,INFRA 2021 Application List.pdf,2026-11-18T17:43:38Z,146143,NaN,None,11885b9c0819,None,None
26,applicants_dot_infra,INFRA 2020 Application List.pdf,2026-11-18T17:43:42Z,155928,NaN,None,311be8b1b766,None,None


## 2. OpenFEMA HMA Subapplications v2

National, every program and status. One row per subapplication (FEMA's unit), including
Not Selected / Not Approved. Subapplications that states screened out and never forwarded
to FEMA are absent (T0.4; asked for in FEMA FOIA Part B).

In [3]:
snap = openfema.read_snapshot()
cat = json.loads((config.ROOT / snap.attrs["snapshot_dir"] / "HmaSubapplications_v2_catalog.json").read_text())
print(snap.attrs["snapshot_dir"], f"{len(snap):,} rows;", "catalog recordCount", cat["recordCount"],
      "| lastRefresh", cat["lastRefresh"], "| lastDataSetRefresh", cat["lastDataSetRefresh"])
snap["program_id"] = openfema.hma_program_id(snap)
snap = snap.join(openfema.id_code(snap["subapplicationIdentifier"]))
snap.groupby("program", dropna=False).agg(rows=("program", "size"), fy_min=("fiscalYear", "min"), fy_max=("fiscalYear", "max")).sort_values("rows", ascending=False)

data/raw/openfema/2026-11-18 63,640 rows; catalog recordCount 63640 | lastRefresh 2026-07-27T22:39:30.327Z | lastDataSetRefresh 2026-11-18T12:14:36.641Z


,rows,fy_min,fy_max
program,,,
Hazard Mitigation Grant Program,42169,1989,2025
Pre-Disaster Mitigation,7922,2000,2026
Flood Mitigation Assistance,5980,1996,2024
Building Resilient Infrastructure and Communities,4871,2020,2025
Hazard Mitigation Grant Program Post Fire,989,2017,2025
Legislative Pre-Disaster Mitigation,414,2008,2010
Flood Mitigation Assistance Swift Current,407,2022,2024
Pre-Disaster Mitigation Congressionally Directed Spending,319,2022,2024
Severe Repetitive Loss,306,2008,2012


### 2a. Check (a): `program` strings for BRIC and PDM, and (b): earmarked PDM and legacy LPDM

ID family = the first letters of `subapplicationIdentifier` (FEMA GO IDs look like
`EMF-2020-BR-143-0016`; eGrants IDs like `PDMC-PJ-09-CA-2019-001` or `LPDM-PJ-10-OR-2024-001`;
NEMIS disaster IDs like `DR-1234-...`).

In [4]:
fam = snap.assign(id_kind=snap["id_code"].where(snap["id_code"] != "", snap["id_family"]))
bp = fam[snap["program"].isin(openfema.BRIC_PDM_PROGRAMS)]
bp.groupby(["program", "id_kind", "program_id"]).agg(rows=("program", "size"), fy_min=("fiscalYear", "min"), fy_max=("fiscalYear", "max"))

rows  fy_min  fy_max
program                                            id_kind program_id                           
Building Resilient Infrastructure and Communities  BR      bric             4871    2020    2025
Legislative Pre-Disaster Mitigation                LPDM    lpdm_legacy       414    2008    2010
Pre-Disaster Mitigation                            DR      pdm_competitive   515    2000    2004
                                                   PD      pdm_cpf           142    2026    2026
                                                   PDMC    pdm_competitive  7265    2003    2019
Pre-Disaster Mitigation Congressionally Directe... LPDM    pdm_cpf           319    2022    2024

In [5]:
# FY2026 "Pre-Disaster Mitigation" records: FEMA GO IDs with code PD, submitted 20-22 July 2026
pd26 = snap[(snap["program"] == openfema.PDM) & (snap["fiscalYear"] == 2026)]
print(len(pd26), "records;", pd26["id_award"].nunique(), "award IDs;", pd26["subapplicantStateAbbreviation"].nunique(), "states")
print("dateSubmittedToFema:", pd26["dateSubmittedToFema"].min(), "to", pd26["dateSubmittedToFema"].max())
print(pd26["status"].value_counts().to_string())
print("federal share $M:", round(pd26["federalShareAmount"].sum() / 1e6, 1))

142 records; 41 award IDs; 40 states
dateSubmittedToFema: 2026-07-06T00:00:00.000Z to 2026-07-22T00:00:00.000Z
status
Subaward Accepted    108
Submitted to FEMA     27
Subaward Offered       7
federal share $M: 167.8


These match FEMA's FY2026 PDM Congressionally Directed Spending NOFO (DHS-26-MT-143-00-01,
ALN **97.143**, applications due 22 July 2026, $189.7M for 125 named projects). So FY2026
earmarks carry the plain program string "Pre-Disaster Mitigation"; `program_id` separates
them by ID code PD and year ≥ 2022. The FY2022–24 earmarks use the program string
"Pre-Disaster Mitigation Congressionally Directed Spending" and eGrants `LPDM-` IDs, the
same prefix as the FY2008–10 "Legislative Pre-Disaster Mitigation" records.

In [6]:
# Status by program_id (outcome field) for the BRIC/PDM family
bp = snap[snap["program_id"] != ""]
pd.crosstab(bp["status"].fillna("(blank)"), bp["program_id"], margins=True)

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf,All
status,,,,,
Approved,0,0,5,0,5
Closed,110,278,4476,8,4872
Closeout Initiated,140,0,0,0,140
Ineligible,1339,0,0,0,1339
Not Approved / Denied,0,131,2447,16,2594
Not Selected,597,0,686,0,1283
Obligated,0,1,153,290,444
Pending,0,4,0,4,8
Revision Requested,0,0,3,1,4


In [7]:
# BRIC selectionStatus (FEMA's national selection outcome) by fiscal year
bric = snap[snap["program_id"] == "bric"]
pd.crosstab(bric["selectionStatus"].fillna("(blank)"), bric["fiscalYear"], margins=True)

fiscalYear,2020,2021,2022,2023,2024,2025,All
selectionStatus,,,,,,,
(blank),0,0,0,0,97,954,1051
Did Not Meet HMA Requirements,156,208,154,194,0,0,712
Identified for Further Review,408,371,518,720,0,0,2017
Not Selected,423,210,129,321,0,0,1083
Withdrawn,6,0,2,0,0,0,8
All,993,789,803,1235,97,954,4871


### 2b. Rows by program × FY × state (BRIC/PDM family), and the CA/OR/WA subset

In [8]:
by_fy = pd.crosstab(bp["fiscalYear"], bp["program_id"], margins=True)
by_fy

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf,All
fiscalYear,,,,,
2000,0,0,176,0,176
2002,0,0,148,0,148
2003,0,0,581,0,581
2004,0,0,91,0,91
2005,0,0,821,0,821
2006,0,0,230,0,230
2007,0,0,471,0,471
2008,0,207,485,0,692
2009,0,101,485,0,586


In [9]:
pac = bp[bp["subapplicantStateAbbreviation"].isin(PACIFIC)]
pd.crosstab([pac["program_id"], pac["fiscalYear"]], pac["subapplicantStateAbbreviation"], margins=True)

subapplicantStateAbbreviation   CA   OR   WA   All
program_id      fiscalYear                        
bric            2020            42    6   36    84
                2021            25    7   27    59
                2022            37   22   29    88
                2023            29   32   29    90
                2024             4    0   34    38
                2025            17   29   26    72
lpdm_legacy     2008            19    0   11    30
                2009            10    0    2    12
                2010            11    0    1    12
pdm_competitive 2003            38   10   13    61
                2005           140   18   28   186
                2006            15    7   10    32
                2007            53    9    6    68
                2008            72    7   13    92
                2009            51    5   14    70
                2010            52    6    8    66
                2011            18    2   13    33
                2012            43    5    8    56
                2013             6    5    7    18
                2014            18    4   15    37
                2015            15    6   11    32
                2016            26    8    8    42
                2017            32    4   31    67
                2018            34    9   32    75
                2019            41    4   26    71
pdm_cpf         2022             3    2    4     9
                2023            11    0    5    16
                2024            13    2    4    19
                2026            16    0    7    23
All                            891  209  458  1558

In [10]:
# Full program x FY x state table for every state (long form; saved nowhere, shown summarised)
long = bp.groupby(["program_id", "fiscalYear", "subapplicantStateAbbreviation"]).size().rename("rows").reset_index()
print(f"{len(long):,} program x FY x state cells; {long['subapplicantStateAbbreviation'].nunique()} states/territories")
long.pivot_table(index="subapplicantStateAbbreviation", columns="program_id", values="rows", aggfunc="sum", fill_value=0).loc[PACIFIC]

1,274 program x FY x state cells; 56 states/territories


program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf
subapplicantStateAbbreviation,,,,
CA,154,40,654,43
OR,96,0,109,4
WA,181,14,243,20


### 2c. Fields present (share non-blank, %), by program_id

Identifiers: UEI, name, city, ZIP, benefiting counties, type. Amounts:
`totalSubapplicationAmount` and `federalShareAmount` are requested/approved;
`totalObligatedAmount` / `totalPaymentAmount` are filled only once a subaward is accepted.

In [11]:
FIELDS = ["subapplicantUei", "subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties",
          "subapplicantType", "subapplicationTitle", "primaryActivity", "totalSubapplicationAmount", "federalShareAmount",
          "totalObligatedAmount", "totalPaymentAmount", "benefitCostRatio", "status", "selectionStatus", "dateSubmittedToFema"]
win = bp[bp["fiscalYear"] >= config.STUDY_FY_START]
(win.groupby("program_id")[FIELDS].agg(lambda s: s.notna().mean() * 100).round(0).T)

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf
subapplicantUei,97.0,0.0,0.0,30.0
subapplicantName,100.0,100.0,100.0,100.0
subapplicantCity,100.0,100.0,100.0,100.0
subapplicantZipCode,100.0,100.0,100.0,100.0
benefitingCounties,100.0,76.0,79.0,88.0
subapplicantType,100.0,100.0,100.0,100.0
subapplicationTitle,80.0,0.0,0.0,0.0
primaryActivity,100.0,100.0,100.0,100.0
totalSubapplicationAmount,100.0,100.0,100.0,100.0
federalShareAmount,100.0,100.0,100.0,100.0


### 2d. Known quirks

- `benefitingCounties` gives county names without states, and some Pacific records list
  counties from other states (e.g. "CLATSOP; WARREN", "SAN MATEO; PLEASANTS; GASTON").
  The coastal flag therefore matches names only within the subapplicant's own state.
- `subapplicantCity` sometimes holds a street address (e.g. "3322 236th Street NE").
- PDM FY2010–19 and LPDM records have no title, UEI or `selectionStatus`.
- FEMA: the financial fields "are not derived from FEMA's official financial systems".
- The catalog `lastRefresh` (2026-07-27) predates `lastDataSetRefresh` (2026-11-18); the
  record count has not changed since T0.4 (63,640).

In [12]:
# Stray county names: Pacific BRIC/PDM records whose counties are not coastal in their own state
ext = openfema.pacific_extract(snap)
other = ext["counties_other"].dropna().str.split(";").explode().str.strip()
print("county names on Pacific records that are not coastal counties of the record's state (top 12):")
print(other.str.upper().str.replace(" COUNTY", "").value_counts().head(12).to_string())
addr_city = ext["subapplicantCity"].fillna("").str.match(r"^\d+\s")
print("\nPacific records whose city field starts with a street number:", int(addr_city.sum()))

county names on Pacific records that are not coastal counties of the record's state (top 12):
counties_other
                  929
SAN BERNARDINO     76
WASHINGTON         59
RIVERSIDE          57
CLACKAMAS          51
MULTNOMAH          45
ST. LOUIS          35
BENTON             34
FRANKLIN           32
DOUGLAS            29
GRANT              27
CLARK              26

Pacific records whose city field starts with a street number: 1


## 3. Check (c): T1.1's 97.047 award-ID rule against OpenFEMA

T1.1 classified USAspending 97.047 awards by FEMA award ID: `BR` = bric, `PC` =
pdm_competitive, `PD` with round ≥ FY2022 = pdm_cpf, earlier `PD` = pdm_other. FEMA GO award
IDs can be joined to OpenFEMA subapplication IDs (exactly, or on the first four parts).
eGrants-era awards cannot.

In [13]:
usa_dir = config.RAW_DIR / "usaspending"
f97 = sorted((usa_dir).glob("*/aln_97.047/*/Assistance_PrimeAwardSummaries_*.csv"))
print(f97[-1].relative_to(config.ROOT))
prime = pd.read_csv(f97[-1], dtype=str, low_memory=False)
cc = openfema.crosscheck_award_ids(prime, snap)
pd.crosstab([cc["fema_program_code"], cc["t11_program_id"]], cc["of_join"], margins=True)

data/raw/usaspending/2026-11-18/aln_97.047/PrimeAwardTransactionsAndSubawards_2026-11-18_H14M23S12859327/Assistance_PrimeAwardSummaries_2026-11-18_H14M23S16_1.csv


of_join                            award_prefix  exact  none   All
fema_program_code t11_program_id                                  
BR                bric                      189    646     0   835
CA                other_97047                 0      0     2     2
FM                other_97047                 2      0     0     2
GR                other_97047                 0      0     1     1
PC                pdm_competitive             0      0   358   358
PD                pdm_cpf                     0      0    64    64
                  pdm_other                   0      0    13    13
All                                         191    646   438  1275

In [14]:
joined = cc[cc["of_join"] != "none"]
print("Joined awards, T1.1 class vs OpenFEMA program_id ('' = an HMA program outside BRIC/PDM):")
print(joined.groupby(["t11_program_id", "of_program_id"]).size().to_string())

Joined awards, T1.1 class vs OpenFEMA program_id ('' = an HMA program outside BRIC/PDM):
t11_program_id  of_program_id
bric            bric             835
other_97047                        2


In [15]:
# PD awards from FY2022 on vs OpenFEMA earmark records, by year (state counts and dollars)
amt = lambda s: round(pd.to_numeric(s, errors="coerce").sum() / 1e6, 1)
usa_pd = cc[cc["t11_program_id"] == "pdm_cpf"].groupby("fema_round_fy").agg(
    usa_awards=("award_id_fain", "size"), usa_states=("recipient_state_code", "nunique"),
    usa_obligated_m=("total_obligated_amount", amt))
of_cpf = snap[snap["program_id"] == "pdm_cpf"].groupby("fiscalYear").agg(
    of_records=("program_id", "size"), of_states=("subapplicantStateAbbreviation", "nunique"),
    of_fed_share_m=("federalShareAmount", amt))
usa_pd.join(of_cpf, how="outer")

,usa_awards,usa_states,usa_obligated_m,of_records,of_states,of_fed_share_m
fema_round_fy,,,,,,
2022,29.0,27.0,144.1,70,27,134.7
2023,34.0,32.0,217.6,127,32,222.9
2024,1.0,1.0,5.7,122,36,180.4
2026,NaN,NaN,NaN,142,40,167.8


In [16]:
# PC awards: FAIN year vs description (eGrants-era PDM; cannot be joined to OpenFEMA)
pc = cc[cc["fema_program_code"] == "PC"]
desc = pc["prime_award_base_transaction_description"].fillna("").str.upper()
pd.crosstab(pc["fema_round_fy"], desc.str.contains("BRIC").map({True: "described as BRIC", False: "PDM / other"}), margins=True)

prime_award_base_transaction_description,PDM / other,described as BRIC,All
fema_round_fy,,,
2008,1,0,1
2010,3,0,3
2011,5,0,5
2012,7,0,7
2013,1,0,1
2014,16,0,16
2015,22,0,22
2016,47,0,47
2017,52,0,52


In [17]:
pc[desc.str.contains("BRIC")][["award_id_fain", "recipient_name", "recipient_state_code", "total_obligated_amount"]]

,award_id_fain,recipient_name,recipient_state_code,total_obligated_amount
462,EMD-2018-PC-0008,TURTLE MOUNTAIN BAND OF CHIPPEWA INDIANS,ND,-1.00
594,EMD-2022-PC-0001,NORTHERN CHEYENNE TRIBE,MT,71471.26
718,EMF-2022-PC-0001,BLUE LAKE RANCHERIA,CA,562356.00
719,EMF-2022-PC-0002,KARUK TRIBE HOUSING AUTHORITY,CA,810885.24
734,EMF-2024-PC-0001,PAUMA BAND OF MISSION INDIANS,CA,126000.00
1103,EMS-2021-PC-0013,TWIN HILLS VILLAGE COUNCIL,AK,179854.00
1104,EMS-2021-PC-0014,NEW STUYAHOK VILLAGE,AK,0.00
1112,EMS-2022-PC-0001,PUYALLUP TRIBE OF INDIANS,WA,167143.99
1113,EMS-2022-PC-0002,MAKAH INDIAN TRIBE OF THE MAKAH INDIAN RESERVATION,WA,108116.00


## 4. PDM FY2010–19 linkability

Titles and UEIs are blank. What else can identify the applicant?

In [18]:
pdm = snap[(snap["program_id"] == "pdm_competitive") & snap["fiscalYear"].between(2010, 2019)]
pdm_pac = pdm[pdm["subapplicantStateAbbreviation"].isin(PACIFIC)]
ID_FIELDS = ["subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties", "subapplicantType",
             "subapplicantUei", "subapplicationTitle", "primaryActivity"]
fill = pd.DataFrame({"national": pdm[ID_FIELDS].notna().mean() * 100, "CA/OR/WA": pdm_pac[ID_FIELDS].notna().mean() * 100}).round(0)
print(len(pdm), "national records;", len(pdm_pac), "in CA/OR/WA")
fill

4292 national records; 497 in CA/OR/WA


,national,CA/OR/WA
subapplicantName,100.0,100.0
subapplicantCity,100.0,100.0
subapplicantZipCode,100.0,100.0
benefitingCounties,79.0,77.0
subapplicantType,100.0,100.0
subapplicantUei,0.0,0.0
subapplicationTitle,0.0,0.0
primaryActivity,100.0,100.0


In [19]:
print(pdm_pac["subapplicantType"].value_counts().to_string())
print()
print(pdm_pac["status"].value_counts().to_string())
print("\ndistinct subapplicant names (CA/OR/WA):", pdm_pac["subapplicantName"].str.upper().str.strip().nunique())
pdm_pac[["fiscalYear", "subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties", "primaryActivity", "status"]].sample(8, random_state=3)

subapplicantType
Local Government                 252
Indian Tribal Government         136
State Government                  54
Special Governmental District     48
Private Non-Profit                 4
other                              3

status
Closed                   281
Not Approved / Denied    137
Not Selected              60
Obligated                 19

distinct subapplicant names (CA/OR/WA): 310


,fiscalYear,subapplicantName,subapplicantCity,subapplicantZipCode,benefitingCounties,primaryActivity,status
49316,2013,City of Yucaipa,Yucaipa,92399,SAN BERNARDINO,Retrofit,Closed
54669,2011,Spokane County,Spokane,99217,SPOKANE,New plan,Closed
5645,2010,Tahoe Truckee Unified School District,Truckee,96161,None,Other,Not Selected
45344,2012,City of Paso Robles,Paso Robles,93446,SAN LUIS OBISPO,New plan,Closed
27635,2014,Ventura County Sheriff's Office of Emergency Services,Ventura,93009,VENTURA; MILLARD; ANDREW,New plan,Closed
60587,2017,Stillaguamish Tribe of Indians,Arlington,98223,SNOHOMISH,Management costs,Closed
53117,2016,California Governors Office of Emergency Services,Mather,95655,None,Management costs,Closed
3151,2018,City of Seattle,Seattle,98104,KING,Retrofit,Closed


In [20]:
# Harbor-related wording is only available through primaryActivity / subactivity (no titles)
acts = pdm_pac["primaryActivity"].value_counts()
print(acts.head(15).to_string())

primaryActivity
New plan                                 127
Retrofit                                  72
Management costs                          62
Plan update                               51
Utility and infrastructure protection     43
Flood control                             39
Other                                     37
Generator                                 29
Wildfire management                       15
Acquisition                                8
Stabilization and restoration              6
Elevation                                  4
Saferoom/shelter                           3
Relocation                                 1


## 5. Extract for T1.3: `data/processed/bric_pacific_subapplicants.csv`

CA/OR/WA BRIC and PDM subapplications (competitive and earmarked, all years, all statuses),
with UEI, name, county, title, status, amounts, `any_coastal_county` (a listed benefiting
county is in `coastal_counties.csv`, matched within the subapplicant's state) and
`zip_in_coastal_county` (subapplicant ZIP → county via the Census ZCTA file from T1.1). Dollar columns are nominal and end in `_nominal`.

In [21]:
ext = openfema.process(None)

snapshot data/raw/openfema/2026-11-18: 63,640 rows
wrote data/processed/bric_pacific_subapplicants.csv: 1,558 rows
     program_id subapplicantStateAbbreviation  rows  coastal  with_uei
           bric                            CA   154       89       154
           bric                            OR    96       38        95
           bric                            WA   181      158       179
    lpdm_legacy                            CA    40       30         0
    lpdm_legacy                            WA    14        9         0
pdm_competitive                            CA   654      360         0
pdm_competitive                            OR   109       33         0
pdm_competitive                            WA   243      160         0
        pdm_cpf                            CA    43       33        16
        pdm_cpf                            OR     4        2         0
        pdm_cpf                            WA    20       14         6


In [22]:
ext["coastal_any"] = ext["any_coastal_county"] | ext.get("zip_in_coastal_county", False)
summ = ext.groupby(["program_id", "subapplicantStateAbbreviation"]).agg(
    rows=("program_id", "size"), in_window=("in_study_window", "sum"),
    coastal_county=("any_coastal_county", "sum"), coastal_county_or_zip=("coastal_any", "sum"),
    with_uei=("has_uei", "sum"), with_title=("has_title", "sum"))
summ = summ.astype(int)
pd.concat([summ, summ.sum().to_frame(("total", "")).T])

rows  in_window  coastal_county  coastal_county_or_zip  with_uei  with_title
bric            CA   154        154              89                     89       154         137
                OR    96         96              38                     38        95          67
                WA   181        181             158                    159       179         155
lpdm_legacy     CA    40         11              30                     31         0           0
                WA    14          1               9                     12         0           0
pdm_competitive CA   654        285             360                    414         0           0
                OR   109         53              33                     40         0           0
                WA   243        159             160                    198         0           0
pdm_cpf         CA    43         43              33                     33        16           0
                OR     4          4               2                      2         0           0
                WA    20         20              14                     17         6           0
total               1558       1007             926                   1033       450         359

## 6. DOT and MARAD lists: common setup

All 13 files were saved by hand in a browser on 2026-11-18 and registered with
`applicants --import-dir` (`"method": "browser"` in the manifest): the DOT and MARAD sites
answer scripted requests with HTTP 403. **Rounds vs fiscal years:** DOT/MARAD name rounds
by the appropriation year (e.g. "PIDP 2019", "TIGER 2010", "INFRA FY2023-24"). Here
`round_fy` = the year in the round label (first year for multi-year rounds). That is not
always the federal fiscal year in which applications were filed or awards obligated;
T2.5b/T2.2 should map with `harborfund.config.fiscal_year()` on dates when they join awards.

In [23]:
import re

RAW = config.RAW_DIR / "applicants"


def latest(source, name):
    d = sorted((RAW / source).glob("*/"))[-1]
    return d / name


STATE_ABBR = {"California": "CA", "Oregon": "OR", "Washington": "WA"}


def round_fy(label):
    m = re.search(r"(20\d\d)", str(label))
    return int(m.group(1)) if m else None

## 7. MARAD PIDP eligible applications, FY2019–2026

Unit: eligible applications (ineligible ones are not listed). No outcome, UEI, city,
county or applicant type.

In [24]:
pidp = pd.read_excel(latest("marad_pidp", "FY19-26 PIDP Eligible Applicants.xlsx"))
print("raw column names:", [repr(c) for c in pidp.columns])  # 'Applicant Name ' has a trailing space
pidp.columns = pidp.columns.str.strip()
pidp["round_fy"] = pidp["Round"].map(round_fy)
print(len(pidp), "rows; columns:", list(pidp.columns))
print("blank cells per column:", pidp.isna().sum().to_dict())
pd.crosstab(pidp["Round"], pidp["State"].where(pidp["State"].isin(PACIFIC), "other"), margins=True)

raw column names: ["'Round'", "'Applicant Name '", "'Project Name'", "'State'", "'PIDP Request'", "'Project Cost'"]
1040 rows; columns: ['Round', 'Applicant Name', 'Project Name', 'State', 'PIDP Request', 'Project Cost', 'round_fy']
blank cells per column: {'Round': 0, 'Applicant Name': 0, 'Project Name': 0, 'State': 0, 'PIDP Request': 0, 'Project Cost': 0, 'round_fy': 0}


State,CA,OR,WA,other,All
Round,,,,,
PIDP 2019,6,0,4,48,58
PIDP 2020,4,1,4,54,63
PIDP 2021,8,2,10,87,107
PIDP 2022,11,3,9,118,141
PIDP 2023,13,5,7,128,153
PIDP 2024,8,7,16,127,158
PIDP 2025,12,10,19,145,186
PIDP 2026,14,4,17,139,174
All,76,32,86,846,1040


In [25]:
# Requests ($M) by round, all states and CA/OR/WA
pidp.assign(pac=pidp["State"].isin(PACIFIC)).groupby("Round").agg(
    apps=("Round", "size"), request_m=("PIDP Request", lambda s: round(s.sum() / 1e6, 1)),
    pacific_apps=("pac", "sum"),
    pacific_request_m=("PIDP Request", lambda s: round(s[pidp.loc[s.index, "State"].isin(PACIFIC)].sum() / 1e6, 1)))

,apps,request_m,pacific_apps,pacific_request_m
Round,,,,
PIDP 2019,58,1198.6,10,216.0
PIDP 2020,63,968.4,9,143.5
PIDP 2021,107,1298.5,20,314.8
PIDP 2022,141,2801.8,23,393.2
PIDP 2023,153,2809.7,25,590.1
PIDP 2024,158,2985.7,31,468.2
PIDP 2025,186,3066.2,41,497.1
PIDP 2026,174,2740.0,35,412.9


In [26]:
# Quirks: (1) row 1 has applicant and project swapped; (2) 2019-23 count vs the ICR's 511
print(pidp.iloc[0][["Round", "Applicant Name", "Project Name", "State"]].to_dict())
short = pidp["Project Name"].str.match(r"^(City|County|Town|Port) of [^,]+(, [A-Z]{2})?$") & (pidp["Project Name"].str.split().str.len() <= 4)
print("rows whose project name is only a place/agency name (swap or placeholder):")
print(pidp[short][["Round", "Applicant Name", "Project Name", "State"]].to_string())
print("\nPIDP 2019-2023 applications in the list:", int(pidp["round_fy"].between(2019, 2023).sum()), "(the ICR cites 511)")
dups = pidp.duplicated(["Round", "Applicant Name", "Project Name"], keep=False)
print("exact duplicate round/applicant/project rows:", int(dups.sum()))

{'Round': 'PIDP 2019', 'Applicant Name': 'Agricultural Maritime Export Facility', 'Project Name': 'City of Milwaukee', 'State': 'WI'}
rows whose project name is only a place/agency name (swap or placeholder):
        Round                         Applicant Name                Project Name State
0   PIDP 2019  Agricultural Maritime Export Facility           City of Milwaukee    WI
6   PIDP 2019                        City of Antioch         City of Antioch, CA    CA
8   PIDP 2019                 City of Conneaut, Ohio  Port of Conneaut Connector    OH
67  PIDP 2020                 City of Conneaut, Ohio  Port of Conneaut Connector    OH

PIDP 2019-2023 applications in the list: 522 (the ICR cites 511)
exact duplicate round/applicant/project rows: 0


In [27]:
pidp[pidp["State"].isin(PACIFIC)].sort_values(["State", "round_fy"])[["Round", "State", "Applicant Name", "Project Name", "PIDP Request"]].head(25)

,Round,State,Applicant Name,Project Name,PIDP Request
6,PIDP 2019,CA,City of Antioch,"City of Antioch, CA",21851509.04
9,PIDP 2019,CA,City of Los Angeles-Harbor Department (Port of Los Angeles),Port of Los Angeles National Multimodal Freight Network Improvement Program: Fenix Container Terminal Intermodal Railyard Expansion and Modernization Project,18184743.00
35,PIDP 2019,CA,Oxnard Harbor District - The Port of Hueneme,Port of Hueneme's Structure for Transfer of Automobiles Creating Key Economic Development (STACKED) Intermodal Project,53380913.00
47,PIDP 2019,CA,Port of Long Beach,Alameda Corridor South Access: Terminal Island Rail Junction Project,16000000.00
48,PIDP 2019,CA,Port of Oakland,Big Ship Ready - Phase 2 Project Component 1: High-Strength Bollards and Fenders Component 2: Crane Relocation and Removal,18392000.00
50,PIDP 2019,CA,Port of Stockton,Rail Bridge Replacement Project,18000000.00
71,PIDP 2020,CA,City of Los Angeles-Harbor Department (Port of Los Angeles),SR 47-Vincent Thomas Bridge & Harbor Boulevard-Front Street Interchange Improvement Project,10000000.00
95,PIDP 2020,CA,Oxnard Harbor District - The Port of Hueneme,Structure for Transfer of Automobiles Creating Key Economic Development_x000D_ (STACKED) Intermodal Project,53380913.00
107,PIDP 2020,CA,Port of Long Beach,East Basin Rail Gateway Expansion: Fourth Track at Ocean Project,8000000.00
111,PIDP 2020,CA,Port of San Francisco,Pier Improvements to Promote Economic Recovery and Sustainability Project,5218133.00


## 8. DOT TIGER (2009) – BUILD (2025 Round 2) eligible applications

Unit: eligible applications. Fields: round, project, applicant, state, city, project type,
urban/rural, request, cost, applicant type, "Project of Merit" (a DOT review rating, not
an award). No outcome, UEI or county.

In [28]:
tig = pd.read_excel(latest("dot_tiger_build", "TIGER 2009 - BUILD 2025 Eligible Applications.xlsx"))
lead = tig["State"].str.match(r"^\s")
print("State values with leading spaces, by round:", tig.loc[lead, "Round"].value_counts().to_dict())
tig["State"] = tig["State"].str.strip()
tig["st"] = tig["State"].map(STATE_ABBR).fillna("other")
tig["round_fy"] = tig["Round"].map(round_fy)
print(len(tig), "rows; columns:", list(tig.columns))
print("blank cells per column:", tig.isna().sum()[lambda s: s > 0].to_dict())
order = tig.groupby("Round")["round_fy"].first().sort_values().index
pd.crosstab(tig["Round"], tig["st"], margins=True).reindex(list(order) + ["All"])

State values with leading spaces, by round: {'BUILD 2018': 851}
14621 rows; columns: ['Round', 'Project Name', 'Applicant Name', 'State', 'City', 'Project Type', 'Urban / Rural', 'Funding Request', 'Project Cost', 'Applicant Type', 'Project of Merit', 'st', 'round_fy']
blank cells per column: {'City': 22, 'Project Type': 1, 'Urban / Rural': 2, 'Applicant Type': 19, 'Project of Merit': 14020}


st,CA,OR,WA,other,All
Round,,,,,
TIGER 2009,120,21,50,1175,1366
TIGER 2010,172,42,70,1355,1639
TIGER 2011,96,19,34,684,833
TIGER 2012,87,16,42,563,708
TIGER 2013,71,15,22,475,583
TIGER 2014,99,16,39,644,798
TIGER 2015,60,10,24,533,627
TIGER 2016,51,8,25,501,585
TIGER 2017,37,8,24,383,452


In [29]:
# Maritime project-type applications in CA/OR/WA by round
mar = tig[tig["Project Type"].str.strip().str.lower().eq("maritime")]
print("Maritime applications nationally:", len(mar), "| CA/OR/WA:", int((mar["st"] != "other").sum()))
pd.crosstab(mar["Round"], mar["st"]).reindex(order).fillna(0).astype(int).query("CA + OR + WA > 0")

Maritime applications nationally: 948 | CA/OR/WA: 164


st,CA,OR,WA,other
Round,,,,
TIGER 2009,7,3,3,81
TIGER 2010,3,2,6,63
TIGER 2011,4,2,3,54
TIGER 2012,5,2,5,45
TIGER 2013,3,1,5,45
TIGER 2014,6,1,4,39
TIGER 2015,3,3,4,40
TIGER 2016,3,2,3,46
TIGER 2017,5,1,2,29


In [30]:
# Quirk: applicant-type labels change across rounds; project-type spellings vary
lab = tig.groupby("Applicant Type")["round_fy"].agg(["min", "max", "size"]).sort_values("size", ascending=False)
print(len(lab), "distinct applicant-type labels; per round:", tig.groupby("Round")["Applicant Type"].nunique().to_dict())
print(tig["Project Type"].value_counts().to_string())
lab.head(25)

36 distinct applicant-type labels; per round: {'BUILD 2018': 12, 'BUILD 2019': 12, 'BUILD 2020': 10, 'BUILD 2025 (Round 2)': 13, 'RAISE 2021': 11, 'RAISE 2022': 12, 'RAISE 2023': 13, 'RAISE 2024': 13, 'RAISE 2025 (Round 1)': 10, 'TIGER 2009': 14, 'TIGER 2010': 8, 'TIGER 2011': 9, 'TIGER 2012': 8, 'TIGER 2013': 14, 'TIGER 2014': 11, 'TIGER 2015': 10, 'TIGER 2016': 10, 'TIGER 2017': 10}
Project Type
Road                      8645
Transit                   1707
Maritime                   948
Planning                   888
Bicycle and Pedestrian     681
Rail                       531
Bicycle-Pedestrian         400
Freight Rail               371
Other                      158
BicyclePedestrian          113
Passenger Rail             108
Aviation                    61
road                         9


,min,max,size
Applicant Type,,,
C: City or Township Government,2013,2025,4203
Local Govt.,2009,2012,2579
B: County Government,2013,2025,2044
A: State Government,2013,2025,1360
D: Special District Government,2013,2025,750
State Govt.,2009,2012,645
I: Indian/Native American Tribal Government (Federally Recognized),2013,2025,532
E: Regional Organization,2013,2025,509
X: Other (specify),2013,2025,455


In [31]:
print("Project of Merit = Yes:", int((tig["Project of Merit"] == "Yes").sum()), "rows, rounds:",
      sorted(tig.loc[tig["Project of Merit"] == "Yes", "Round"].unique()))

Project of Merit = Yes: 601 rows, rounds: ['BUILD 2025 (Round 2)', 'RAISE 2022', 'RAISE 2023', 'RAISE 2024']


## 9. DOT INFRA FY2022+ spreadsheets (with award outcomes)

The INFRA files cover the combined MPDG application (INFRA, Mega, Rural). Unlike PIDP
and TIGER–BUILD they carry **`Awarded`** and **`AwardedAmount`**, so the outcome is in the
list. FY2022 comes as two files: DOT's "INFRA FY22 Applications" (all 329 applications,
incl. 78 marked "Not eligible") and the INFRA sheet of the FY2022 all-programs list (the
251 eligible ones, with a zero "Awarded Amount" column).

In [32]:
INFRA_FILES = {
    "INFRA FY2022": "INFRA FY22 Applications.xlsx",
    "INFRA FY2023-24": "INFRA FY23-24 Applications.xlsx",
    "INFRA FY2025-26": "INFRA Applications 25-26.xlsx",
    "INFRA FY2026": "INFRA FY26 Applications (1).xlsx",
}
frames = []
for lab, fn in INFRA_FILES.items():
    d = pd.read_excel(latest("dot_infra", fn))
    d = d[d["ApplicantState"].notna()]  # drop blank/footer rows
    d["round"] = lab
    d["file"] = fn
    frames.append(d)
infra = pd.concat(frames, ignore_index=True)
infra["st"] = infra["ApplicantState"].str[:2]
infra["eligible"] = infra[["Eligibility", "EligibleForINFRA"]].bfill(axis=1).iloc[:, 0]
infra["awarded_any"] = infra["Awarded"].notna()
print({lab: list(f.columns) for lab, f in zip(INFRA_FILES, frames)})
pd.crosstab([infra["round"], infra["eligible"].fillna("(blank)")], infra["st"].where(infra["st"].isin(PACIFIC), "other"), margins=True)

{'INFRA FY2022': ['GrantTrackingNumber', 'ApplicantState', 'UrbanRural', 'ApplicantOrganizationName', 'ProjectName', 'Awarded', 'AwardedAmount', 'TotalFundsRequested', 'FutureEligibleProjectCosts', 'Eligibility', 'round', 'file'], 'INFRA FY2023-24': ['ApplicantState', 'UrbanRural', 'ApplicantOrganizationName', 'ProjectName', 'Awarded', 'AwardedAmount', 'TotalFundsRequested', 'FutureEligibleProjectCosts', 'EligibleForINFRA', 'round', 'file'], 'INFRA FY2025-26': ['ApplicantState', 'UrbanRural', 'ApplicantOrganizationName', 'ProjectName', 'Awarded', 'AwardedAmount', 'TotalFundsRequested', 'FutureEligibleProjectCosts', 'EligibleForINFRA', 'round', 'file'], 'INFRA FY2026': ['GrantTrackingNumber', 'ApplicantState', 'UrbanRural', 'ApplicantOrganizationName', 'ProjectName', 'Awarded', 'AwardedAmount', 'TotalFundsRequested', 'FutureEligibleProjectCosts', 'Eligibility', 'round', 'file']}


st                                                  CA  OR  WA  other  All
round           eligible                                                  
INFRA FY2022    Not eligible                         6   0   1     71   78
                Potentially or partially eligible   38   6   2    205  251
INFRA FY2023-24 Potentially or partially eligible   26   4   3    158  191
INFRA FY2025-26 Potentially or partially eligible   24   3   4    143  174
INFRA FY2026    Not eligible                         0   0   0      6    6
                Potentially or partially eligible   17   1   2     62   82
All                                                111  14  12    645  782

In [33]:
infra.groupby("round").agg(
    apps=("round", "size"), awarded=("awarded_any", "sum"),
    request_m=("TotalFundsRequested", lambda s: round(s.sum() / 1e6)),
    awarded_m=("AwardedAmount", lambda s: round(pd.to_numeric(s, errors="coerce").sum() / 1e6)),
    pacific_apps=("st", lambda s: s.isin(PACIFIC).sum()),
    pacific_awarded=("awarded_any", lambda s: s[infra.loc[s.index, "st"].isin(PACIFIC)].sum()))

,apps,awarded,request_m,awarded_m,pacific_apps,pacific_awarded
round,,,,,,
INFRA FY2022,329,38,28975,2265,53,4
INFRA FY2023-24,191,41,24719,4352,33,4
INFRA FY2025-26,174,51,24891,4519,31,8
INFRA FY2026,88,26,4922,627,20,2


In [34]:
print(infra.groupby("round")["Awarded"].value_counts(dropna=False).to_string())
usd22 = pd.read_excel(latest("dot_infra", "FY 2022 USDOT Discretionary Grant Applicant List.xlsx"), sheet_name="INFRA")
f22 = infra[infra["round"] == "INFRA FY2022"]
key = lambda d: (d["ApplicantOrganizationName"].str.strip() + "|" + d["ProjectName"].str.strip())
print("\nFY2022: all-programs INFRA sheet", len(usd22), "rows;", "INFRA FY22 file eligible rows",
      int((f22["eligible"] != "Not eligible").sum()), "; sheet rows found in the FY22 file:", int(key(usd22).isin(key(f22)).sum()))
print("FY2026 file vs FY2025-26 file: same applicant+project in both:", int(key(infra[infra["round"] == "INFRA FY2026"]).isin(key(infra[infra["round"] == "INFRA FY2025-26"])).sum()))

round            Awarded             
INFRA FY2022     NaN                     291
                 Yes - INFRA              26
                 Yes- Mega                 6
                 Yes- Rural                6
INFRA FY2023-24  NaN                     150
                 Yes- INFRA               26
                 Yes- Mega                 7
                 Yes - Rural               6
                 Yes- INFRA and Mega       2
INFRA FY2025-26  NaN                     123
                 Yes - INFRA              33
                 Yes - Rural               8
                 Yes - Mega                7
                 Yes - INFRA and Mega      3
INFRA FY2026     NaN                      62
                 Yes                      26

FY2022: all-programs INFRA sheet 251 rows; INFRA FY22 file eligible rows 251 ; sheet rows found in the FY22 file: 251
FY2026 file vs FY2025-26 file: same applicant+project in both: 4


In [35]:
infra[infra["st"].isin(PACIFIC)].sort_values(["round", "st"])[["round", "st", "ApplicantOrganizationName", "ProjectName", "TotalFundsRequested", "Awarded", "AwardedAmount"]]

,round,st,ApplicantOrganizationName,ProjectName,TotalFundsRequested,Awarded,AwardedAmount
15,INFRA FY2022,CA,California Department of Transportation,I-5 Managed Lanes,2.110000e+08,NaN,0.0
16,INFRA FY2022,CA,California Department of Transportation,Otay Mesa East Port of Entry Project,1.700000e+08,Yes - INFRA,150000000.0
17,INFRA FY2022,CA,California Department of Transportation,POLA Supply Chain Project,8.946000e+07,NaN,0.0
18,INFRA FY2022,CA,California Department of Transportation,Watsonville-Santa Cruz Multimodal Corridor Program,3.000000e+07,Yes- Mega,30000000.0
19,INFRA FY2022,CA,California Department of Transportation,Pit River Bridge,3.000000e+07,NaN,0.0
20,INFRA FY2022,CA,California Department of Transportation,State Route 58 Truck Climbing Lane,4.400000e+07,NaN,0.0
21,INFRA FY2022,CA,California Department of Transportation - Caltrans,Interstate 5 Stockton Channel Viaduct Bridge Rehabilitation Project,1.000000e+08,NaN,0.0
22,INFRA FY2022,CA,City of Arroyo Grande,US 101 Brisco-Halcyon Interchange Modification Project,2.940000e+07,NaN,0.0
23,INFRA FY2022,CA,City of Menifee,Scott Road/Bundy Canyon Road Widening Project,2.500000e+07,NaN,0.0
24,INFRA FY2022,CA,City of Murrieta,I-215 and Keller Road Interchange Project,2.280000e+07,NaN,0.0


## 10. DOT FASTLANE/INFRA FY2016–2021 PDFs

Read with `harborfund.ingest.infra_pdfs` (needs poppler's `pdftotext`). Fields: row
number, project name, applicant organization, state, request and one or two cost columns
(the columns differ by year). No outcome, UEI, city or applicant type. State, request and
cost are read reliably (row counts equal the last row number in every file); project and
applicant names are kept together in `name_text`, and wrapped lines can land on the
neighbouring row.

In [36]:
from harborfund.ingest import infra_pdfs

if infra_pdfs.have_pdftotext():
    pdfs = infra_pdfs.read_all(latest("dot_infra", ""))
    pdfs["round_fy"] = pdfs["round"].map(round_fy)
    chk = pdfs.groupby("round").agg(rows=("round", "size"), last_row_no=("row_no", "max"),
                                    no_state=("state", lambda s: s.isna().sum()),
                                    no_request=("request", lambda s: s.isna().sum()),
                                    request_m=("request", lambda s: round(s.sum() / 1e6)))
    print(chk.to_string())
    out = pd.crosstab(pdfs["round"], pdfs["state"].where(pdfs["state"].isin(PACIFIC), "other"), margins=True)
else:
    pdfs = None
    out = "pdftotext not found: install poppler to profile the FY2016-21 PDFs"
out

                 rows  last_row_no  no_state  no_request  request_m
round                                                              
FASTLANE FY2016   212          212         0           0       9797
FASTLANE FY2017   195          195         0           0       7163
INFRA FY2017-18   234          234         0           0      12286
INFRA FY2019      193          193         0           0       9766
INFRA FY2020      173          173         0           0       7376
INFRA FY2021      157          157         0           0       6886


state,CA,OR,WA,other,All
round,,,,,
FASTLANE FY2016,34,6,13,159,212
FASTLANE FY2017,21,4,17,153,195
INFRA FY2017-18,36,4,13,181,234
INFRA FY2019,31,4,8,150,193
INFRA FY2020,26,2,9,136,173
INFRA FY2021,26,1,14,116,157
All,174,21,74,895,1164


In [37]:
if pdfs is not None:
    port = pdfs["name_text"].str.contains(r"\b(?:Port|Harbor|Marine|Terminal|Maritime)\b", case=False)
    print("CA/OR/WA rows whose names mention port/harbor/marine/terminal/maritime:",
          int((port & pdfs["state"].isin(PACIFIC)).sum()))
    show = pdfs[port & pdfs["state"].isin(PACIFIC)][["round", "state", "name_text", "request"]]
else:
    show = None
show

CA/OR/WA rows whose names mention port/harbor/marine/terminal/maritime: 47


,round,state,name_text,request
16,FASTLANE FY2016,CA,Global Opportunities at the Port of Oakland (GoPort) Commission,140034000.0
23,FASTLANE FY2016,CA,I-15/Muscatel Street Interchange City of Hesperia Port of Los Angeles Zero Emission/Truck Trip City of Los Angeles Harbor Department,42600000.0
24,FASTLANE FY2016,CA,Reduction/Freight Efficiency Project (Port of Los Angeles),35100000.0
32,FASTLANE FY2016,CA,Calexico East Land Port of Entry Expansion Project Commission Los Angeles County Metropolitan,30000000.0
40,FASTLANE FY2016,CA,Middle Harbor Terminal Redevelopment Project Port of Long Beach Green Port Gateway Phase II — Terminal Island Wye,90000000.0
41,FASTLANE FY2016,CA,Port of Long Beach Rail Improvements County of San Bernardino DBA San,13800000.0
152,FASTLANE FY2016,OR,Coos Bay Rail Line - Tunnel Rehabilitation Project Oregon International Port of Coos Bay,11000000.0
153,FASTLANE FY2016,OR,Portland Marine Terminal Freight and Jobs Access Port of Portland Project,7329000.0
202,FASTLANE FY2016,WA,Port of Tacoma Road/Interstate 5 Interchange City of Fife Reconstruction,21000000.0
203,FASTLANE FY2016,WA,South Terminal Intermodal Modernization Project Port of Everett,20000000.0


## 11. Summary for the coverage note

One row per source. `outcome` = whether the list itself says who won.

In [38]:
rows = [
    ("PIDP (MARAD)", "FY2019-26", "eligible applications", len(pidp), int(pidp["State"].isin(PACIFIC).sum()), "no"),
    ("TIGER-BUILD-RAISE (DOT)", "2009-2025 R2", "eligible applications", len(tig), int((tig["st"] != "other").sum()), "no ('Project of Merit' only)"),
    ("INFRA FY2022+ (DOT xlsx)", "FY2022-26", "applications (FY22 incl. not eligible)", len(infra), int(infra["st"].isin(PACIFIC).sum()), "yes (Awarded, AwardedAmount)"),
    ("FASTLANE/INFRA FY2016-21 (DOT pdf)", "FY2016-21", "applications", len(pdfs) if pdfs is not None else None,
     int(pdfs["state"].isin(PACIFIC).sum()) if pdfs is not None else None, "no"),
    ("BRIC (OpenFEMA)", "FY2020-25", "subapplications submitted to FEMA", int((snap["program_id"] == "bric").sum()),
     int(((snap["program_id"] == "bric") & snap["subapplicantStateAbbreviation"].isin(PACIFIC)).sum()), "yes (status, selectionStatus)"),
    ("PDM competitive (OpenFEMA)", "FY2010-19", "subapplications submitted to FEMA",
     int(((snap["program_id"] == "pdm_competitive") & snap["fiscalYear"].between(2010, 2019)).sum()),
     int(((snap["program_id"] == "pdm_competitive") & snap["fiscalYear"].between(2010, 2019) & snap["subapplicantStateAbbreviation"].isin(PACIFIC)).sum()), "yes (status)"),
]
pd.DataFrame(rows, columns=["source", "years", "unit", "rows", "CA/OR/WA rows", "outcome in list"])

,source,years,unit,rows,CA/OR/WA rows,outcome in list
0,PIDP (MARAD),FY2019-26,eligible applications,1040,194,no
1,TIGER-BUILD-RAISE (DOT),2009-2025 R2,eligible applications,14621,2464,no ('Project of Merit' only)
2,INFRA FY2022+ (DOT xlsx),FY2022-26,applications (FY22 incl. not eligible),782,137,"yes (Awarded, AwardedAmount)"
3,FASTLANE/INFRA FY2016-21 (DOT pdf),FY2016-21,applications,1164,269,no
4,BRIC (OpenFEMA),FY2020-25,subapplications submitted to FEMA,4871,431,"yes (status, selectionStatus)"
5,PDM competitive (OpenFEMA),FY2010-19,subapplications submitted to FEMA,4292,497,yes (status)
